# Concept Recovery Analysis - multi-koncept x multi-metoda

Pipeline jednoprzejazdowy. Dla kazdej pary `(concept, method)`:

1. Wyciaga aktywacje CLS z warstwy `DEBIAS_LAYER` (bez wczesniejszego debiasingu).
2. Iteracyjnie (`N_ITER` razy) wyznacza CAV i rzutuje. Konwencja: `iter k = stan po k rzutach`.
3. Wstrzykuje finalnie zdebiesowane aktywacje z powrotem do CLIP w `DEBIAS_LAYER`.
4. Reszta sieci dziala bez debiasingu - zbierane sa CLS-y z warstw `DEBIAS_LAYER+1 .. NUM_LAYERS-1`.
5. Na kazdej z tych warstw trenowana jest LR (CV po C) - wykres "powrot konceptu".

Konfiguruj listy `CONCEPTS` i `METHODS`. Model laduje sie raz; ekstrakcja `X_raw` jest robiona raz per koncept (3 metody dziela ja).

In [ ]:
import os, sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from dotenv import load_dotenv
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from transformers import AutoModel, AutoImageProcessor
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')
load_dotenv()

In [ ]:
CONCEPTS      = ['eyeglasses']                       # lista konceptow do przetworzenia
METHODS       = ['diff_means', 'lr', 'pclarc']       # lista metod debiasingu
DEBIAS_LAYER  = 15                                   # warstwa, na ktorej iteracyjnie debiasujemy
N_ITER        = 7                                    # liczba iteracji CAV + rzutowania

NUM_LAYERS     = 24
GPU_BATCH_SIZE = 32
NUM_WORKERS    = 0
MODEL_ID       = 'openai/clip-vit-large-patch14'

# Regresja logistyczna (CAV i recovery): CV po C (L2)
LR_CS          = [0.01, 0.1]
LR_CV_FOLDS    = 5

RECOVERY_LAYERS = list(range(DEBIAS_LAYER + 1, NUM_LAYERS))

assert 0 <= DEBIAS_LAYER < NUM_LAYERS - 1, \
    f'DEBIAS_LAYER musi byc w [0, {NUM_LAYERS - 2}]'
assert all(m in ('diff_means', 'lr', 'pclarc') for m in METHODS), \
    "METHODS: kazdy element musi byc z {'diff_means','lr','pclarc'}"
assert len(CONCEPTS) > 0 and len(METHODS) > 0, 'CONCEPTS i METHODS musza byc niepuste'

print(f'Koncepty : {CONCEPTS}')
print(f'Metody   : {METHODS}')
print(f'Warstwa  : {DEBIAS_LAYER}  |  Iteracje: {N_ITER}')
print(f'Recovery : {RECOVERY_LAYERS}')

In [ ]:
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def concept_paths(concept):
    """Zwraca (metadata_path, images_dir) dla konceptu."""
    return (
        ROOT / 'data' / f'metadata_{concept}.csv',
        ROOT / 'data' / f'images_{concept}',
    )


def run_dir(concept, method):
    """Katalog wyjsciowy dla pary (concept, method)."""
    run_id = f'layer{DEBIAS_LAYER:02d}_{method}_iter{N_ITER}'
    return ROOT / 'data' / 'checkpoint2_data' / concept / 'recovery' / run_id


# Walidacja: dla kazdego konceptu musza istniec metadata i folder ze zdjeciami
for concept in CONCEPTS:
    metadata_path, images_dir = concept_paths(concept)
    if not metadata_path.exists():
        raise FileNotFoundError(
            f'Brak metadanych dla {concept!r}: {metadata_path}\n'
            f"Uruchom scripts/download_data.ipynb z CONCEPT={concept!r}"
        )
    if not images_dir.exists():
        raise FileNotFoundError(f'Brak folderu zdjec dla {concept!r}: {images_dir}')

print(f'ROOT : {ROOT}')

In [ ]:
# Podglad splitow per koncept (przed wlasciwym uruchomieniem pipeline'u)
for concept in CONCEPTS:
    metadata_path, _ = concept_paths(concept)
    df_meta = pd.read_csv(metadata_path)
    if concept not in df_meta.columns:
        raise ValueError(
            f"Brak kolumny {concept!r} w {metadata_path}. Dostepne: {list(df_meta.columns)}"
        )
    df_train = df_meta[df_meta['split'] == 'train']
    df_test  = df_meta[df_meta['split'] == 'test']
    print(
        f'{concept:20s}: '
        f'train={len(df_train):4d} ({df_train[concept].mean():.1%} pos) | '
        f'test={len(df_test):3d} ({df_test[concept].mean():.1%} pos)'
    )

In [ ]:
HF_TOKEN = os.getenv('HF_TOKEN')
device   = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype    = torch.bfloat16 if device == 'cuda' else torch.float32

processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
).to(device).eval()

print(f'Model: {MODEL_ID} | {device} | {dtype}')


class CelebADataset(Dataset):
    def __init__(self, df, images_dir, processor, label_col):
        self.df = df.reset_index(drop=True)
        self.images_dir = Path(images_dir)
        self.processor  = processor
        self.label_col  = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.images_dir / row['filename']).convert('RGB')
        px  = self.processor(images=img, return_tensors='pt').pixel_values.squeeze(0)
        return px, int(row[self.label_col])


def make_loader(df, images_dir, label_col):
    return DataLoader(
        CelebADataset(df, images_dir, processor, label_col),
        batch_size=GPU_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device == 'cuda'),
        persistent_workers=(NUM_WORKERS > 0),
    )

In [ ]:
def _validate_xy(X_tr, y_tr, where):
    if not np.isfinite(X_tr).all():
        n_bad = int((~np.isfinite(X_tr)).sum())
        raise ValueError(
            f'{where}: X_tr zawiera {n_bad} wartosci NaN/Inf '
            f'(prawdopodobnie kumulacja bledu numerycznego w iteracjach).'
        )
    n_pos = int((y_tr == 1).sum())
    n_neg = int((y_tr == 0).sum())
    if n_pos == 0 or n_neg == 0:
        raise ValueError(
            f'{where}: jedna z klas pusta (pos={n_pos}, neg={n_neg}). '
            f'Sprawdz kodowanie etykiet (musi byc 0/1).'
        )


def compute_diff_means(X_tr, y_tr):
    """CAV = znormalizowana roznica srednich klas. Kierunek: neg -> pos."""
    _validate_xy(X_tr, y_tr, 'compute_diff_means')
    X = X_tr.astype(np.float64)
    diff = X[y_tr == 1].mean(axis=0) - X[y_tr == 0].mean(axis=0)
    n = np.linalg.norm(diff)
    if n < 1e-10:
        raise ValueError(
            'compute_diff_means: srednie klas niemal identyczne - '
            'koncept nieseparowalny liniowo na tej warstwie/iteracji.'
        )
    return (diff / n).astype(np.float32)


def compute_lr_cav(X_tr, y_tr):
    """CAV z LR z L2 i CV-strojonym C. Wagi przeskalowane do przestrzeni oryginalnej."""
    _validate_xy(X_tr, y_tr, 'compute_lr_cav')
    sc = StandardScaler()
    Xs = sc.fit_transform(X_tr.astype(np.float64))
    lr = LogisticRegressionCV(
        Cs=LR_CS, cv=LR_CV_FOLDS, scoring='accuracy',
        solver='lbfgs', max_iter=2000, random_state=42, n_jobs=-1,
    )
    lr.fit(Xs, y_tr)
    w = lr.coef_[0] / sc.scale_
    n = np.linalg.norm(w)
    if n < 1e-10:
        raise ValueError(
            'compute_lr_cav: wagi LR (po deskalizacji) maja zerowa norme.'
        )
    return (w / n).astype(np.float32), sc, lr


def fit_dm_threshold(X_tr, y_tr, cav):
    """Prog na trainie: punkt srodkowy miedzy srednimi projekcji klas."""
    proj = X_tr.astype(np.float64) @ cav.astype(np.float64)
    return float((proj[y_tr == 1].mean() + proj[y_tr == 0].mean()) / 2)


def cav_accuracy_dm(X, y, cav, threshold):
    """Accuracy z progu wyznaczonego na trainie. CAV diff_means kieruje neg->pos."""
    proj = X.astype(np.float64) @ cav.astype(np.float64)
    return accuracy_score(y, (proj > threshold).astype(int))


def cav_accuracy_lr(X, y, sc, lr_m):
    return lr_m.score(sc.transform(X.astype(np.float64)), y)


def apply_orthogonal(X, cav):
    X64 = X.astype(np.float64)
    c64 = cav.astype(np.float64)
    return (X64 - np.outer(X64 @ c64, c64)).astype(np.float32)


def apply_pclarc(X, cav, target_val):
    X64 = X.astype(np.float64)
    c64 = cav.astype(np.float64)
    return (X64 + np.outer(target_val - X64 @ c64, c64)).astype(np.float32)

In [ ]:
class _StopForward(Exception):
    pass


def extract_layer(loader, layer_idx):
    """Zwraca (X: float32 (N, D), y: int (N,)) - CLS token z `layer_idx`."""
    _buf = []

    def _hook(module, input, output):
        hidden = output[0] if isinstance(output, tuple) else output
        _buf.append(hidden[:, 0, :].detach().float().cpu().numpy())
        raise _StopForward()

    handle = model.vision_model.encoder.layers[layer_idx].register_forward_hook(_hook)
    X_all, y_all = [], []
    try:
        with torch.no_grad():
            for pixels, labels in loader:
                _buf.clear()
                try:
                    model.get_image_features(pixel_values=pixels.to(device, dtype=dtype))
                except _StopForward:
                    pass
                if _buf:
                    X_all.append(_buf[0])
                y_all.extend(labels.tolist())
    finally:
        handle.remove()

    return np.concatenate(X_all, axis=0), np.array(y_all, dtype=int)


def run_with_injection(loader, inject_layer, debiased_cls, recovery_layers):
    """Forward pass: w `inject_layer` podmienia CLS na `debiased_cls`,
    zbiera CLS z warstw `recovery_layers`. Zwraca dict {layer -> (N, D)}."""
    encoder_layers = model.vision_model.encoder.layers
    captures = {k: [] for k in recovery_layers}
    ptr = [0]

    def inject_hook(module, input, output):
        is_tuple = isinstance(output, tuple)
        hidden   = output[0] if is_tuple else output
        B        = hidden.shape[0]
        cls_deb  = torch.from_numpy(
            debiased_cls[ptr[0]:ptr[0] + B]
        ).to(hidden.device, dtype=hidden.dtype)
        ptr[0] += B
        h_new = hidden.clone()
        h_new[:, 0, :] = cls_deb
        return (h_new,) + output[1:] if is_tuple else h_new

    def make_cap(k):
        def hook(module, input, output):
            hidden = output[0] if isinstance(output, tuple) else output
            captures[k].append(hidden[:, 0, :].detach().float().cpu().numpy())
        return hook

    handles = [encoder_layers[inject_layer].register_forward_hook(inject_hook)]
    for k in recovery_layers:
        handles.append(encoder_layers[k].register_forward_hook(make_cap(k)))

    try:
        with torch.no_grad():
            for pixels, _ in loader:
                model.get_image_features(pixel_values=pixels.to(device, dtype=dtype))
    finally:
        for h in handles:
            h.remove()

    return {k: np.concatenate(v, axis=0) for k, v in captures.items()}

In [ ]:
def iter_debias(X_tr_raw, y_tr, X_te_raw, y_te, method, n_iter):
    """Iteracyjny debiasing. Konwencja: iter k = stan po k zastosowanych rzutach.
    Zwraca (X_tr_deb, X_te_deb, df_iter)."""
    X_tr_deb = X_tr_raw.copy()
    X_te_deb = X_te_raw.copy()

    iter_records = []
    prev_cav, prev_target = None, None

    for it in range(n_iter + 1):
        # iter > 0: zastosuj rzut z CAV poprzedniej iteracji
        if it > 0:
            if method == 'pclarc':
                X_tr_deb = apply_pclarc(X_tr_deb, prev_cav, prev_target)
                X_te_deb = apply_pclarc(X_te_deb, prev_cav, prev_target)
            else:
                X_tr_deb = apply_orthogonal(X_tr_deb, prev_cav)
                X_te_deb = apply_orthogonal(X_te_deb, prev_cav)

        # CAV + accuracy (prog/sklasyfikator z trainu, oceniane na trainie i teście)
        if method == 'lr':
            cav, sc_it, lr_it = compute_lr_cav(X_tr_deb, y_tr)
            acc_tr = cav_accuracy_lr(X_tr_deb, y_tr, sc_it, lr_it)
            acc_te = cav_accuracy_lr(X_te_deb, y_te, sc_it, lr_it)
        else:
            cav = compute_diff_means(X_tr_deb, y_tr)
            threshold = fit_dm_threshold(X_tr_deb, y_tr, cav)
            acc_tr = cav_accuracy_dm(X_tr_deb, y_tr, cav, threshold)
            acc_te = cav_accuracy_dm(X_te_deb, y_te, cav, threshold)

        # Cache CAV i target_val (pclarc) do nastepnej iteracji
        if method == 'pclarc':
            prev_target = float(
                (X_tr_deb.astype(np.float64)[y_tr == 0] @ cav.astype(np.float64)).mean()
            )
        else:
            prev_target = None
        prev_cav = cav

        iter_records.append({'iteration': it, 'train_acc': acc_tr, 'test_acc': acc_te})

    return X_tr_deb, X_te_deb, pd.DataFrame(iter_records)


def train_recovery(caps_tr, y_tr, caps_te, y_te, recovery_layers):
    """Trenuje LR (CV po C) na CLS-ach z `recovery_layers`. Zwraca DataFrame."""
    rows = []
    for k in recovery_layers:
        X_tr_k, X_te_k = caps_tr[k], caps_te[k]
        _validate_xy(X_tr_k, y_tr, f'recovery L{k}')
        sc = StandardScaler()
        Xs_tr = sc.fit_transform(X_tr_k.astype(np.float64))
        Xs_te = sc.transform(X_te_k.astype(np.float64))
        lr = LogisticRegressionCV(
            Cs=LR_CS, cv=LR_CV_FOLDS, scoring='accuracy',
            solver='lbfgs', max_iter=2000, random_state=42, n_jobs=-1,
        )
        lr.fit(Xs_tr, y_tr)
        rows.append({
            'layer_id':  k,
            'train_acc': lr.score(Xs_tr, y_tr),
            'test_acc':  lr.score(Xs_te, y_te),
            'best_C':    float(lr.C_[0]),
        })
    return pd.DataFrame(rows)

In [ ]:
all_results = {}  # {(concept, method): {'iter': df, 'recovery': df, 'data_out': Path}}

for concept in tqdm(CONCEPTS, desc='Koncepty'):
    metadata_path, images_dir = concept_paths(concept)
    df_meta  = pd.read_csv(metadata_path)
    df_train = df_meta[df_meta['split'] == 'train'].reset_index(drop=True)
    df_test  = df_meta[df_meta['split'] == 'test'].reset_index(drop=True)

    loader_train = make_loader(df_train, images_dir, concept)
    loader_test  = make_loader(df_test,  images_dir, concept)

    # Ekstrakcja raz na koncept (3 metody dziela `X_raw`)
    print(f'\n[{concept}] Ekstrakcja warstwy {DEBIAS_LAYER}...')
    X_tr_raw, y_tr = extract_layer(loader_train, DEBIAS_LAYER)
    X_te_raw, y_te = extract_layer(loader_test,  DEBIAS_LAYER)
    print(f'  train: {X_tr_raw.shape} | test: {X_te_raw.shape}')

    for method in tqdm(METHODS, desc=f'{concept}', leave=False):
        data_out = run_dir(concept, method)
        data_out.mkdir(parents=True, exist_ok=True)

        # 1) Iteracyjny debiasing
        X_tr_deb, X_te_deb, df_iter = iter_debias(
            X_tr_raw, y_tr, X_te_raw, y_te, method, N_ITER
        )
        df_iter.to_csv(data_out / 'iter_debiasing_accuracy.csv', index=False)

        # 2) Forward pass z wstrzykieciem zdebiesowanego CLS
        caps_tr = run_with_injection(loader_train, DEBIAS_LAYER, X_tr_deb, RECOVERY_LAYERS)
        caps_te = run_with_injection(loader_test,  DEBIAS_LAYER, X_te_deb, RECOVERY_LAYERS)

        # 3) Recovery: LR (CV po C) per warstwa
        df_rec = train_recovery(caps_tr, y_tr, caps_te, y_te, RECOVERY_LAYERS)
        df_rec.to_csv(data_out / 'layer_recovery_accuracy.csv', index=False)

        all_results[(concept, method)] = {
            'iter':     df_iter,
            'recovery': df_rec,
            'data_out': data_out,
        }
        baseline = df_iter.iloc[0]['test_acc']
        after    = df_iter[df_iter['iteration'] == N_ITER]['test_acc'].values[0]
        max_rec  = df_rec['test_acc'].max()
        print(
            f'  [{method:10s}] test: baseline={baseline:.3f} -> '
            f'po {N_ITER} iter={after:.3f} | max recovery={max_rec:.3f}'
        )

print('\nWszystko gotowe.')

## Wykresy

In [ ]:
METHOD_COLORS = {'diff_means': '#e41a1c', 'lr': '#377eb8', 'pclarc': '#4daf4a'}
METHOD_LABELS = {'diff_means': 'Diff Means', 'lr': 'LR', 'pclarc': 'PCLARC'}

for concept in CONCEPTS:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # lewy: iter accuracy (test) - 3 metody na jednym wykresie
    ax = axes[0]
    baseline_te = all_results[(concept, METHODS[0])]['iter'].iloc[0]['test_acc']
    for method in METHODS:
        df_iter = all_results[(concept, method)]['iter']
        ax.plot(df_iter['iteration'], df_iter['test_acc'],
                marker='o', ms=5, color=METHOD_COLORS[method], label=METHOD_LABELS[method])
    ax.axhline(0.5, color='gray', ls=':', lw=1.2, label='Random')
    ax.set_xlabel('Iteracja debiasingu (= liczba zastosowanych rzutow)', fontsize=13)
    ax.set_ylabel('Test accuracy CAV', fontsize=13)
    ax.set_title(
        f'Iteracyjny debiasing - {concept}, warstwa {DEBIAS_LAYER}',
        fontsize=13, fontweight='bold'
    )
    ax.set_xticks(range(N_ITER + 1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.05))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.01))
    ax.grid(True, which='major', alpha=0.4)
    ax.grid(True, which='minor', alpha=0.15)
    ax.legend(fontsize=11)

    # prawy: recovery (test) - 3 metody na jednym wykresie
    ax = axes[1]
    for method in METHODS:
        df_rec = all_results[(concept, method)]['recovery']
        ax.plot(df_rec['layer_id'], df_rec['test_acc'],
                marker='o', ms=5, color=METHOD_COLORS[method], label=METHOD_LABELS[method])
    ax.axhline(baseline_te, color='red', ls=':', lw=1.5,
               label=f'Baseline (warstwa {DEBIAS_LAYER}, 0 iter)')
    ax.axhline(0.5, color='gray', ls=':', lw=1.2, label='Random')
    ax.set_xlabel('Warstwa CLIP', fontsize=13)
    ax.set_ylabel('Test accuracy (recovery LR)', fontsize=13)
    ax.set_title(
        f'Powrot konceptu po debiasingu warstwy {DEBIAS_LAYER}',
        fontsize=13, fontweight='bold'
    )
    ax.set_xticks(RECOVERY_LAYERS)
    ax.set_xticklabels([str(k) for k in RECOVERY_LAYERS], rotation=90)
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.05))
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.01))
    ax.grid(True, which='major', alpha=0.4)
    ax.grid(True, which='minor', alpha=0.15)
    ax.legend(fontsize=11)

    plt.suptitle(f'Concept Recovery - {concept}', fontsize=16, y=1.02)
    plt.tight_layout()

    out_path = (
        ROOT / 'data' / 'checkpoint2_data' / concept / 'recovery'
        / f'comparison_layer{DEBIAS_LAYER:02d}_iter{N_ITER}.png'
    )
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Zapisano: {out_path}')

In [ ]:
summary_rows = []
for (concept, method), d in all_results.items():
    df_iter = d['iter']
    df_rec  = d['recovery']
    baseline = float(df_iter.iloc[0]['test_acc'])
    after    = float(df_iter[df_iter['iteration'] == N_ITER]['test_acc'].values[0])
    max_rec  = float(df_rec['test_acc'].max())
    max_layer = int(df_rec.loc[df_rec['test_acc'].idxmax(), 'layer_id'])
    summary_rows.append({
        'concept':            concept,
        'method':             method,
        'baseline_test':      round(baseline, 3),
        f'after_{N_ITER}iter_test': round(after, 3),
        'max_recovery_test':  round(max_rec, 3),
        'max_recovery_layer': max_layer,
    })

df_summary = pd.DataFrame(summary_rows)
summary_path = ROOT / 'data' / 'checkpoint2_data' / f'recovery_summary_layer{DEBIAS_LAYER:02d}_iter{N_ITER}.csv'
df_summary.to_csv(summary_path, index=False)
print(f'Zapisano: {summary_path}
')
print(df_summary.to_string(index=False))